# Taller — Abandono de producto financiero
## Regresión logística en Python · Clase 9

**Curso:** Machine Learning aplicado a Economía y Finanzas · FCEA · 2026-2

**Estudiante:** Sebastián Rojas

Una entidad de banca de personas quiere pasar de la tasa agregada de abandono a una **probabilidad individual**. Cada fila de `abandono_producto_financiero.csv` es un cliente observado una vez (corte transversal) con su desenlace ya ocurrido: `abandono = 1` si cerró la relación con el banco y `abandono = 0` si permanece.

Es la misma familia de modelos que en la Clase 8 (crédito alemán), con otro desenlace:

$$
P(\texttt{abandono}=1 \mid x) = \Lambda(x\beta) = \frac{e^{x\beta}}{1+e^{x\beta}} \in (0,1)
$$

Los coeficientes del logit están en **log-odds**; $e^{\beta_j}$ es el *odds ratio*. OR $>1$: factor de **riesgo** (más abandono). OR $<1$: factor **protector**.

Las respuestas a las preguntas 1–21 están en celdas Markdown debajo de cada bloque de código, con las cifras de esta corrida.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from patsy import build_design_matrices
from scipy import stats
from scipy.stats import chi2_contingency

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = False

CSV = "abandono_producto_financiero.csv"

if not Path(CSV).exists():
    raise FileNotFoundError(f"No está {CSV}. Déjelo junto a este notebook.")

# 1. Estructura de los datos

In [2]:
banco = pd.read_csv(CSV, encoding="utf-8")
banco.head()

,numero_fila,id_cliente,apellido,puntaje_crediticio,pais,sexo,edad,antiguedad,saldo,numero_productos,tiene_tarjeta,miembro_activo,salario_estimado,abandono
0,1,15634602,Hargrave,619,Francia,Mujer,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,España,Mujer,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,Francia,Mujer,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,Francia,Mujer,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,España,Mujer,43,2,125510.82,1,1,1,79084.10,0


In [3]:
banco.columns.tolist()

['numero_fila',
 'id_cliente',
 'apellido',
 'puntaje_crediticio',
 'pais',
 'sexo',
 'edad',
 'antiguedad',
 'saldo',
 'numero_productos',
 'tiene_tarjeta',
 'miembro_activo',
 'salario_estimado',
 'abandono']

In [4]:
banco.shape

(10000, 14)

In [5]:
banco.dtypes

numero_fila             int64
id_cliente              int64
apellido                  str
puntaje_crediticio      int64
pais                      str
sexo                      str
edad                    int64
antiguedad              int64
saldo                 float64
numero_productos        int64
tiene_tarjeta           int64
miembro_activo          int64
salario_estimado      float64
abandono                int64
dtype: object

In [6]:
banco.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
numero_fila,10000.0,NaN,NaN,NaN,5000.5,2886.89568,1.0,2500.75,5000.5,7500.25,10000.0
id_cliente,10000.0,NaN,NaN,NaN,15690940.5694,71936.186123,15565701.0,15628528.25,15690738.0,15753233.75,15815690.0
apellido,10000,2932,Smith,32,NaN,NaN,NaN,NaN,NaN,NaN,NaN
puntaje_crediticio,10000.0,NaN,NaN,NaN,650.5288,96.653299,350.0,584.0,652.0,718.0,850.0
pais,10000,3,Francia,5014,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sexo,10000,2,Hombre,5457,NaN,NaN,NaN,NaN,NaN,NaN,NaN
edad,10000.0,NaN,NaN,NaN,38.9218,10.487806,18.0,32.0,37.0,44.0,92.0
antiguedad,10000.0,NaN,NaN,NaN,5.0128,2.892174,0.0,3.0,5.0,7.0,10.0
saldo,10000.0,NaN,NaN,NaN,76485.889288,62397.405202,0.0,0.0,97198.54,127644.24,250898.09
numero_productos,10000.0,NaN,NaN,NaN,1.5302,0.581654,1.0,1.0,1.0,2.0,4.0


In [7]:
# Faltantes y filas duplicadas
pd.Series({"valores faltantes": banco.isna().sum().sum(),
           "filas duplicadas": banco.duplicated().sum()})

valores faltantes    0
filas duplicadas     0
dtype: int64

Se quitan los identificadores (`numero_fila`, `id_cliente`, `apellido`): distinguen filas, no explican el abandono. `pais` y `sexo` pasan a `category` con los niveles del diccionario, en ese orden; así `C(pais)` toma **Francia** como referencia y `C(sexo)` toma **Mujer**. `abandono` queda como entero 0/1.

In [8]:
datos = banco.drop(columns=["numero_fila", "id_cliente", "apellido"]).copy()

datos["pais"] = pd.Categorical(datos["pais"], categories=["Francia", "Alemania", "España"])
datos["sexo"] = pd.Categorical(datos["sexo"], categories=["Mujer", "Hombre"])
datos["abandono"] = datos["abandono"].astype(int)

datos.dtypes

puntaje_crediticio       int64
pais                  category
sexo                  category
edad                     int64
antiguedad               int64
saldo                  float64
numero_productos         int64
tiene_tarjeta            int64
miembro_activo           int64
salario_estimado       float64
abandono                 int64
dtype: object

In [9]:
# Niveles (el primero es la referencia) y control de que ningún valor quedó por fuera
print(datos["pais"].cat.categories.tolist())
print(datos["sexo"].cat.categories.tolist())
datos[["pais", "sexo"]].isna().sum()

['Francia', 'Alemania', 'España']
['Mujer', 'Hombre']


pais    0
sexo    0
dtype: int64

In [10]:
cuantitativas = ["puntaje_crediticio", "edad", "antiguedad", "saldo",
                 "numero_productos", "salario_estimado"]
categoricas_binarias = ["pais", "sexo", "tiene_tarjeta", "miembro_activo", "abandono"]

pd.Series({
    "columnas en la base cruda": banco.shape[1],
    "columnas para el modelo": datos.shape[1],
    "predictores (x)": datos.shape[1] - 1,
    "cuantitativas": len(cuantitativas),
    "categóricas o binarias (con abandono)": len(categoricas_binarias),
})

columnas en la base cruda                14
columnas para el modelo                  11
predictores (x)                          10
cuantitativas                             6
categóricas o binarias (con abandono)     5
dtype: int64

### Respuestas — sección 1

**1. Variables en la base cruda y en el modelo.** La base cruda tiene **10 000 filas y 14 columnas**. Al quitar los tres identificadores (`numero_fila`, `id_cliente`, `apellido`) quedan **11 columnas para el modelo**: 10 predictores $x$ y la respuesta `abandono`. No hay valores faltantes ni filas duplicadas.

**2. Cuantitativas.** **6**: `puntaje_crediticio`, `edad`, `antiguedad`, `saldo`, `numero_productos` (conteo entero de 1 a 4, que el diccionario trata como continuo) y `salario_estimado`.

**3. Categóricas o binarias.** **5**: `pais` (3 niveles), `sexo` (2 niveles), `tiene_tarjeta` (0/1), `miembro_activo` (0/1) y la respuesta `abandono` (0/1). En total, 6 + 5 = 11.

**4. Qué implica esta mezcla para el logit en Python.**
- Las cuantitativas entran tal cual: su coeficiente es el cambio en log-odds por **una unidad** (un año, un punto, una unidad monetaria). Por eso el OR de `saldo` o `salario_estimado` va a salir ≈ 1: una unidad monetaria es un cambio mínimo, y conviene leerlos por 10 000 o 50 000.
- Las binarias 0/1 ya son dummies: entran sin `C()` y su coeficiente compara 1 contra 0.
- `pais` y `sexo` son texto. Con `C(pais)` y `C(sexo)`, la fórmula de `statsmodels` (patsy) crea sola $k-1$ dummies y deja como referencia el **primer nivel** de la categoría (Francia y Mujer). Salen 2 coeficientes de país (Alemania, España) y 1 de sexo (Hombre), cada uno leído contra la referencia.
- Con dummies hechas a mano (`pd.get_dummies`) hay que elegir y quitar la referencia (si entran las tres dummies de país junto con el intercepto hay colinealidad perfecta: la trampa de las dummies) y repetir exactamente la misma codificación en test y en los perfiles. Con `C()` y categorías fijas, `predict` aplica sola la misma codificación a datos nuevos.